In [1]:
import torch

from datasets import load_dataset
from torch.utils.data import Dataset

from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.13.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
dataset = load_dataset(
    "imagefolder",
    data_dir=r"C:\Users\praut\project crop doctor\Wheat"
)

print(dataset)
print("Columns:", dataset["train"].column_names)
print("Total images:", len(dataset["train"]))

Resolving data files:   0%|          | 0/2942 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 2942
    })
})
Columns: ['image', 'label']
Total images: 2942


In [4]:
dataset = dataset["train"].train_test_split(
    test_size=0.2,
    seed=42
)

print("Training images:", len(dataset["train"]))
print("Validation images:", len(dataset["test"]))

Training images: 2353
Validation images: 589


In [5]:
labels = dataset["train"].features["label"].names

print("Wheat classes:")

for i, label in enumerate(labels):
    print(i, ":", label)

id2label = {
    i: label for i, label in enumerate(labels)
}

label2id = {
    label: i for i, label in enumerate(labels)
}

print("Number of classes:", len(labels))

Wheat classes:
0 : Wheat___Brown_Rust
1 : Wheat___Healthy
2 : Wheat___Yellow_Rust
Number of classes: 3


In [6]:
model_name = "google/vit-base-patch16-224-in21k"

processor = ViTImageProcessor.from_pretrained(
    model_name
)

print("Processor loaded successfully!")

Processor loaded successfully!


In [7]:
class WheatDataset(Dataset):

    def __init__(self, hf_dataset, processor):
        self.dataset = hf_dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        item = self.dataset[idx]

        image = item["image"].convert("RGB")

        inputs = self.processor(
            images=image,
            return_tensors="pt"
        )

        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "labels": item["label"]
        }

In [8]:
train_dataset = WheatDataset(
    dataset["train"],
    processor
)

eval_dataset = WheatDataset(
    dataset["test"],
    processor
)

print("Wheat datasets created successfully!")

Wheat datasets created successfully!


In [9]:
sample = train_dataset[0]

print(sample.keys())
print("Image shape:", sample["pixel_values"].shape)
print("Label:", sample["labels"])

dict_keys(['pixel_values', 'labels'])
Image shape: torch.Size([3, 224, 224])
Label: 2


In [10]:
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

print("Wheat ViT model created successfully!")

Loading weights:   0%|          | 0/6 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
encoder.layer.{0...11}.layernorm_after.bias             | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.weight              | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.bias          | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.weight | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_before.bias            | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_after.weight           | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.bias                | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.query.weight | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.bias   | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_before.weight          | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.

Wheat ViT model created successfully!


In [11]:
def collate_fn(examples):

    pixel_values = torch.stack([
        example["pixel_values"]
        for example in examples
    ])

    labels_batch = torch.tensor([
        example["labels"]
        for example in examples
    ])

    return {
        "pixel_values": pixel_values,
        "labels": labels_batch
    }

In [12]:
training_args = TrainingArguments(
    output_dir="./wheat-vit-results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=True,

    logging_steps=50,

    report_to="none"
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    data_collator=collate_fn
)

print("Trainer created successfully!")

Trainer created successfully!


In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.248136,0.257758
2,0.142665,0.093593
3,0.102417,0.070810


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=885, training_loss=0.2102821414753542, metrics={'train_runtime': 995.6507, 'train_samples_per_second': 7.09, 'train_steps_per_second': 0.889, 'total_flos': 5.4702085742038426e+17, 'train_loss': 0.2102821414753542, 'epoch': 3.0})